In [1]:
import pandas as pd
from transformers import T5Tokenizer

### Load Data

In [2]:
train_data = pd.read_csv("../data/raw/samsum-train.csv")
val_data = pd.read_csv("../data/raw/samsum-validation.csv")

test_data = pd.read_csv("../data/raw/samsum-test.csv")

In [3]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [4]:
train_data.shape

(14732, 3)

In [5]:
val_data.shape

(818, 3)

In [6]:
train_data = train_data.dropna(subset=["dialogue", "summary"]).reset_index(drop=True)
val_data = val_data.dropna(subset=["dialogue", "summary"]).reset_index(drop=True)

In [7]:
print(f"Shape of training data = {train_data.shape}")
print(f"Shape of validation data = {val_data.shape}")

Shape of training data = (14731, 3)
Shape of validation data = (818, 3)


### Data pre - processing

In [8]:
import re

def clean_data(text):
    text = re.sub(r"<.*?>", " ", text)   # Remove HTML tags
    text = re.sub(r"\r\n|\r|\n", " ", text)  # Remove line breaks
    text = re.sub(r"\s+", " ", text)     # Normalize whitespace
    text = text.strip()

    return text

In [9]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data) 

In [10]:
if test_data is not None:
    test_data["dialogue"] = test_data["dialogue"].apply(clean_data)
    test_data["summary"] = test_data["summary"].apply(clean_data)

# remove rows that became empty after cleaning
train_data = train_data[(train_data["dialogue"] != "") & (train_data["summary"] != "")].reset_index(drop=True)
val_data = val_data[(val_data["dialogue"] != "") & (val_data["summary"] != "")].reset_index(drop=True)

if test_data is not None:
    test_data = test_data[(test_data["dialogue"] != "") & (test_data["summary"] != "")].reset_index(drop=True)

print(train_data.shape, val_data.shape, None if test_data is None else test_data.shape)

(14731, 3) (818, 3) (819, 3)


### Tokenization

In [11]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [12]:
# raw data => tokenized inputs for fine-tuning

def tokenize(data):
    input_texts = [
        "summarize: " + dialogue
        for dialogue in data["dialogue"]
    ]

    inputs = tokenizer(
        input_texts,
        max_length=512,
        truncation=True
    )

    targets = tokenizer(
        text_target=data["summary"],
        max_length=150,
        truncation=True
    )

    inputs["labels"] = targets["input_ids"]

    return inputs

In [13]:
from datasets import Dataset

In [14]:
train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)

In [15]:
test_dataset = Dataset.from_pandas(test_data) if test_data is not None else None

In [16]:
train_dataset[0]

{'id': '13818513',
 'dialogue': "Amanda: I baked cookies. Do you want some? Jerry: Sure! Amanda: I'll bring you tomorrow :-)",
 'summary': 'Amanda baked cookies and will bring Jerry some tomorrow.'}

In [17]:
train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

In [18]:
if test_dataset is not None:
    test_dataset = test_dataset.map(
        tokenize,
        batched=True
    )

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

In [19]:
print(f"Length of training dataset = {len(train_dataset[0]["input_ids"])}")
print(f"Length of labels = {len(train_dataset[0]["labels"])}")

Length of training dataset = 30
Length of labels = 11


In [20]:
print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 14731
})
Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 818
})
Dataset({
    features: ['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 819
})


In [21]:
print(train_dataset[0].keys())

dict_keys(['id', 'dialogue', 'summary', 'input_ids', 'attention_mask', 'labels'])


In [22]:
print(train_dataset[0]["labels"][:30])

[21542, 13635, 5081, 11, 56, 830, 16637, 128, 5721, 5, 1]


In [23]:
import numpy as np

in_lens = np.array([len(x) for x in train_dataset["input_ids"]])
out_lens = np.array([len(x) for x in train_dataset["labels"]])

print(f"input  : mean={in_lens.mean():.0f}  p95={np.percentile(in_lens, 95):.0f}  max={in_lens.max()}  truncated={(in_lens >= 512).mean() * 100:.2f}%")
print(f"summary: mean={out_lens.mean():.0f}  p95={np.percentile(out_lens, 95):.0f}  max={out_lens.max()}  truncated={(out_lens >= 128).mean() * 100:.2f}%")

input  : mean=148  p95=367  max=512  truncated=1.14%
summary: mean=29  p95=59  max=94  truncated=0.00%


In [24]:
print(tokenizer.decode(train_dataset[0]["input_ids"]))
print(tokenizer.decode(train_dataset[0]["labels"]))

summarize: Amanda: I baked cookies. Do you want some? Jerry: Sure! Amanda: I'll bring you tomorrow :-)</s>
Amanda baked cookies and will bring Jerry some tomorrow.</s>


In [25]:
import os

os.makedirs("../data/tokenized", exist_ok=True)

train_dataset.save_to_disk(
    "../data/tokenized/train"
)

val_dataset.save_to_disk(
    "../data/tokenized/val"
)

Saving the dataset (0/1 shards):   0%|          | 0/14731 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/818 [00:00<?, ? examples/s]

In [26]:
if test_dataset is not None:
    test_dataset.save_to_disk(
        "../data/tokenized/test"
    )

Saving the dataset (0/1 shards):   0%|          | 0/819 [00:00<?, ? examples/s]